# Segmentación de Clientes con Metodología RFM
## Análisis de comportamiento y clasificación estratégica de clientes

**Autor:** RobertScience Data Consulting  
**Proyecto:** Avance de Proyecto – Parte 3  
**Enfoque:** Customer Analytics & Unsupervised Learning  
**Entorno:** Python / Jupyter Notebook  

---

## Descripción del proyecto

El presente análisis tiene como objetivo segmentar clientes a partir de sus patrones de compra utilizando la metodología **RFM (Recency, Frequency, Monetary)**.

Esta técnica permite clasificar a los clientes en función de:

- **Recency (R):** Qué tan reciente fue su última compra  
- **Frequency (F):** Con qué frecuencia realizan compras  
- **Monetary (M):** Cuánto dinero han gastado  

A partir de estos indicadores, es posible identificar distintos perfiles de clientes, tales como clientes leales, clientes en riesgo o clientes de alto valor.

Este tipo de análisis es fundamental en entornos empresariales, ya que permite optimizar estrategias de marketing, mejorar la retención de clientes y maximizar el valor del negocio.

## Importación de librerías

En esta sección se importan las librerías necesarias para la manipulación de datos, análisis exploratorio y construcción del modelo de segmentación RFM.

Se utilizan herramientas estándar dentro del ecosistema de ciencia de datos en Python, lo que permite garantizar eficiencia, escalabilidad y buenas prácticas en el análisis.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

## Carga del conjunto de datos

En esta sección se realiza la carga del dataset que contiene información de transacciones de clientes en un entorno retail.

El dataset incluye variables como identificador de cliente, fecha de compra, cantidad de productos y valor monetario, las cuales serán fundamentales para la construcción del modelo RFM.

In [ ]:
df = pd.read_csv(
    r"D:\Documentos\Ebac\Finales\Tarea M25-CD –RobertScience\Avance Proyecto parte 3\M30 Online Retail.csv",
    encoding='latin-1'
)

df.head()

## Exploración inicial de los datos

Antes de aplicar cualquier técnica de segmentación, es necesario analizar la estructura del dataset.

En esta etapa se revisan:

- Tipos de datos
- Valores nulos
- Comportamiento general de las variables

Este análisis permite identificar posibles problemas y preparar adecuadamente los datos para el modelo RFM.

In [ ]:
df.info()

df.describe()

## Análisis de valores nulos

Se identifican las variables que contienen valores faltantes, ya que estos pueden afectar el cálculo correcto de los indicadores RFM.

In [ ]:
df.isnull().sum()

## Limpieza de datos

Para garantizar la calidad del análisis, se realizan los siguientes ajustes:

- Eliminación de registros sin identificador de cliente
- Eliminación de transacciones canceladas
- Eliminación de valores inválidos en cantidad y precio

Esto permite trabajar únicamente con transacciones válidas.

In [ ]:
# Eliminar clientes nulos
df = df.dropna(subset=['CUSTOMER_ID'])

# Eliminar cancelaciones (Invoice que empiezan con C)
df = df[~df['INVOICE_NO'].astype(str).str.startswith('C')]

# Eliminar valores negativos o cero
df = df[df['QUANTITY'] > 0]
df = df[df['UNIT_PRICE'] > 0]

df.head()

## Cálculo del valor monetario

Se calcula el valor total de cada transacción multiplicando la cantidad de productos por el precio unitario.

Esta variable será utilizada para el cálculo del componente Monetary dentro del modelo RFM.

In [ ]:
df['TOTAL_PRICE'] = df['QUANTITY'] * df['UNIT_PRICE']

df.head()

## Conversión de fechas

Se transforma la variable de fecha al formato adecuado para poder realizar cálculos relacionados con el tiempo, como la recencia de compra.

In [ ]:
df['INVOICE_DATE'] = pd.to_datetime(
    df['INVOICE_DATE'],
    dayfirst=True
)

df.info()

## Definición de fecha de referencia

Para calcular la recencia, es necesario definir una fecha de corte que represente el momento actual del análisis.

A partir de esta fecha se calculará cuántos días han pasado desde la última compra de cada cliente.

In [ ]:
# Fecha de referencia (última fecha del dataset + 1 día)
fecha_referencia = df['INVOICE_DATE'].max() + pd.Timedelta(days=1)

fecha_referencia

## Construcción de métricas RFM

Se agrupan los datos por cliente para calcular:

- Recency: días desde la última compra
- Frequency: número de compras realizadas
- Monetary: valor total gastado

Estas métricas permiten analizar el comportamiento individual de cada cliente.

In [ ]:
rfm = df.groupby('CUSTOMER_ID').agg({
    'INVOICE_DATE': lambda x: (fecha_referencia - x.max()).days,
    'INVOICE_NO': 'nunique',
    'TOTAL_PRICE': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm.head()

## Análisis inicial del modelo RFM

Se analizan las métricas obtenidas para comprender la distribución del comportamiento de los clientes.

Esto permite identificar patrones como clientes frecuentes, clientes inactivos o clientes de alto valor.

In [ ]:
rfm.describe()

## Asignación de puntuaciones RFM

Para facilitar la segmentación, se transforman las métricas RFM en puntuaciones.

Cada cliente recibe un score del 1 al 5 en función de su comportamiento:

- Recency: menor valor → mejor score
- Frequency: mayor valor → mejor score
- Monetary: mayor valor → mejor score

Esto permite clasificar a los clientes de forma estandarizada.

In [ ]:
# Recency (invertido: menor es mejor)
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])

# Frequency (mayor es mejor)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])

# Monetary (mayor es mejor)
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

rfm.head()

## Construcción del score RFM

Se combinan las puntuaciones individuales en un único indicador que resume el valor del cliente.

Este score facilita la segmentación y el análisis estratégico.

In [ ]:
rfm['RFM_score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

rfm.head()

## Segmentación de clientes

A partir del score RFM, se clasifican los clientes en distintos segmentos estratégicos.

Esta segmentación permite identificar perfiles como:

- Clientes de alto valor
- Clientes leales
- Clientes en riesgo
- Clientes ocasionales

In [ ]:
def segmentar_cliente(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4:
        return 'Cliente Premium'
    elif row['F_score'] >= 4:
        return 'Cliente Leal'
    elif row['R_score'] <= 2:
        return 'Cliente en Riesgo'
    else:
        return 'Cliente Ocasional'

rfm['Segmento'] = rfm.apply(segmentar_cliente, axis=1)

rfm.head()

## Distribución de segmentos

Se analiza la cantidad de clientes en cada segmento para entender la composición de la base de clientes.

In [ ]:
rfm['Segmento'].value_counts()

## Visualización de segmentos

Se presenta una visualización que permite interpretar de manera clara la distribución de clientes por segmento.

In [ ]:
plt.figure(figsize=(8,5))
rfm['Segmento'].value_counts().plot(kind='bar')

plt.title('Distribución de Segmentos de Clientes')
plt.xlabel('Segmento')
plt.ylabel('Cantidad de Clientes')

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

## Insights clave

A partir del análisis realizado, se identificaron los siguientes puntos estratégicos:

- Los clientes clasificados como **Premium y Leales** representan el segmento de mayor valor, concentrando la mayor frecuencia de compra y contribución económica.

- Existe un grupo relevante de **clientes en riesgo**, caracterizados por una baja recencia, lo que indica la necesidad de implementar estrategias de reactivación.

- Los **clientes ocasionales** presentan una oportunidad de crecimiento, ya que con acciones adecuadas pueden evolucionar hacia segmentos de mayor valor.

- La distribución de segmentos evidencia una base de clientes heterogénea, lo que refuerza la importancia de aplicar estrategias diferenciadas según el perfil de cada cliente.

## Conclusión del análisis

A partir del análisis realizado, se logró segmentar a los clientes utilizando la metodología RFM, lo que permitió identificar distintos patrones de comportamiento dentro de la base de datos.

El cálculo de las métricas de recencia, frecuencia y valor monetario facilitó la construcción de un modelo capaz de clasificar a los clientes en función de su actividad y contribución económica.

La asignación de puntuaciones y la posterior segmentación permitieron diferenciar perfiles estratégicos, destacando clientes de alto valor, clientes leales, así como clientes en riesgo de abandono.

Adicionalmente, la visualización de los segmentos permitió comprender la distribución de la base de clientes, proporcionando una visión clara sobre la composición del negocio.

En conjunto, este análisis demuestra cómo el uso de técnicas de segmentación permite transformar datos transaccionales en información útil para la toma de decisiones, contribuyendo a mejorar estrategias de retención, fidelización y crecimiento de clientes.

---

**RobertScience Data Consulting**  
Data Science | Customer Analytics | Advanced Analytics  
https://robertscience.online/